# 01 - POPE feature extraction with Qwen2-VL-2B (Google Colab)

Requires a GPU; run in Colab (Runtime > Change runtime type > GPU, T4 or better). The 2B model needs about 6-8 GB of GPU memory at batch size 1.

Run the cells top to bottom: Config -> Colab setup -> Dataset inspection -> Token check -> Prompt -> Model load -> Scoring -> Extraction -> Merge -> Verification -> (optional) Download.

Outputs in `OUT_DIR` (default `results/cached_features/`, or the Drive folder if `USE_DRIVE=True`): `pope_full_features.pt` (about 0.8 GB for 9,000 questions), `pope_metadata.parquet`, `pope_run_config.json` and the shard folder `pope_full_shards/` used for resuming. Do not commit the `.pt` file.

If Colab disconnects, re-run from the top with the same config; finished 100-question shards are skipped. On CUDA out-of-memory, restart the session and keep `BATCH_SIZE = 1`.

## Config

In [ ]:
# Config
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
DEVICE = "cuda"
DTYPE = "bfloat16"                            # model compute dtype
STORE_DTYPE = "float16"                       # dtype of saved hidden states

DATASET_NAME = "lmms-lab-encoder/POPE"
DATASET_CONFIG = None                         # None = HF default config
DATASET_LOCAL_PATH = "data/pope"              # loaded from disk if present, else downloaded

POPE_SETTINGS = None                          # None = all settings, or e.g. ["adversarial"]
SUBSET_SIZE = None                            # None = full run; an int takes the first N rows of the seeded order
SHUFFLE_ORDER = True                          # seeded random processing order
SEED = 0
BATCH_SIZE = 1                                # keep 1
CHECKPOINT_EVERY = 100                        # questions per shard

PROMPT_SUFFIX = " Answer yes or no."
SPLIT_FRACS = (0.70, 0.15, 0.15)              # train / dev / test, assigned by image

OUT_DIR = "results/cached_features"           # ignored if USE_DRIVE
FEATURES_NAME = "pope_full_features.pt"
META_NAME = "pope_metadata.parquet"
CONFIG_NAME = "pope_run_config.json"
SHARD_DIRNAME = "pope_full_shards"
KEEP_SHARDS = True

USE_DRIVE = False                             # mount Google Drive and write outputs to DRIVE_DIR
DRIVE_DIR = "/content/drive/MyDrive/sc4001_pope"
DOWNLOAD_AT_END = False                       # download the final files at the end
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"


## a. Colab setup

In [ ]:
# Setup
import os, sys, subprocess, json, hashlib, time, datetime, shutil
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "torch", "torchvision", "transformers>=4.45,<5", "accelerate", "datasets<5",
                       "scikit-learn", "pillow", "numpy", "pandas", "pyarrow", "matplotlib", "seaborn", "tqdm"])

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0),
      "| memory GB: %.1f" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

IN_COLAB = "google.colab" in sys.modules

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "download_datasets.py").exists():
            return q
    return None

ROOT = find_repo_root()
if ROOT is None:
    target = Path("/content/SC4001Assignment_2026") if IN_COLAB else Path.cwd() / "SC4001Assignment_2026"
    if not target.exists():
        r = subprocess.run(["git", "clone", REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    if (target / "src").exists():
        ROOT = target
    else:
        # Repo not reachable; this notebook does not import from src/, so continue in a scratch folder
        ROOT = Path("/content/sc4001_run") if IN_COLAB else Path.cwd() / "sc4001_run"
        ROOT.mkdir(parents=True, exist_ok=True)
        print("WARNING: repo clone failed; using scratch folder", ROOT)
os.chdir(ROOT)
print("Working directory:", ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path(DRIVE_DIR)
else:
    OUT = ROOT / OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
SHARD_DIR = OUT / SHARD_DIRNAME
SHARD_DIR.mkdir(exist_ok=True)
print("Outputs go to:", OUT)

## b. Dataset inspection

The default config of `lmms-lab-encoder/POPE` has one split, `test`, with 9,000 rows and fields `id, question_id, question, answer, image_source, image, category`. The POPE setting (random / popular / adversarial) is the `category` field (3,000 rows each). `question_id` is not unique; `id` is. All 500 COCO images appear in all three settings, so train/dev/test is split by image.

The cell below prints these properties from the loaded data and stops if the structure differs.

In [ ]:
import numpy as np, pandas as pd
from datasets import load_dataset, load_from_disk

local = ROOT / DATASET_LOCAL_PATH
if local.exists() and any(local.iterdir()):
    print("Loading POPE from disk:", local)
    ds = load_from_disk(str(local))
else:
    print("Downloading POPE from Hugging Face ...")
    ds = load_dataset(DATASET_NAME) if DATASET_CONFIG is None else load_dataset(DATASET_NAME, DATASET_CONFIG)
    local.parent.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(local))

print(ds)
print("Split names:", list(ds.keys()))
SPLIT_NAME = "test" if "test" in ds else (list(ds.keys())[0] if len(ds) == 1 else None)
assert SPLIT_NAME is not None, f"Unexpected split structure {list(ds.keys())}; this notebook handles the single-split default config only."
split_ds = ds[SPLIT_NAME]
FIELDS = list(split_ds.features.keys())
print("Using split:", SPLIT_NAME, "| fields:", FIELDS, "| rows:", len(split_ds))
for needed in ["question", "answer", "image", "image_source", "category"]:
    assert needed in FIELDS, f"Expected field '{needed}' missing; got {FIELDS}"

meta0 = split_ds.remove_columns("image").to_pandas()
meta0["row_idx"] = np.arange(len(meta0))
meta0["answer"] = meta0["answer"].str.strip().str.lower()
assert set(meta0["answer"]) <= {"yes", "no"}, set(meta0["answer"])
print("\nPOPE settings ('category'):", meta0["category"].value_counts().to_dict())
print("yes/no balance overall:", meta0["answer"].value_counts().to_dict())
print(pd.crosstab(meta0["category"], meta0["answer"]))
print("unique images:", meta0["image_source"].nunique(), "| images appearing in >1 setting:",
      int((meta0.groupby("image_source")["category"].nunique() > 1).sum()))
print("'id' unique:", meta0["id"].is_unique, "| 'question_id' unique:", meta0["question_id"].is_unique)
print("Row order of first 100 rows by setting:", meta0.head(100)["category"].value_counts().to_dict())

## c. Token check: yes / no token ids

The prompt ends with Qwen's assistant turn header, so the first answer token has no leading space. P(yes) is the summed probability of the single-token ids `Yes, yes, YES` and P(no) that of `No, no, NO`; leading-space variants are printed but not scored. `answer_mass = P(yes)+P(no)` is saved per question, and `top1_token` shows the model's actual first token. The ids are stored in `pope_run_config.json` and a resumed run must match them.

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer

variants = ["Yes", "yes", "YES", " Yes", " yes", " YES", "No", "no", "NO", " No", " no", " NO"]
print(f"{'string':10s} {'token ids':20s} single-token?")
for v in variants:
    ids = tok.encode(v, add_special_tokens=False)
    print(f"{v!r:10s} {str(ids):20s} {len(ids) == 1}")

YES_STRINGS, NO_STRINGS = ["Yes", "yes", "YES"], ["No", "no", "NO"]
def single_ids(strings):
    out = {}
    for s in strings:
        ids = tok.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            out[s] = ids[0]
        else:
            print(f"  dropping {s!r}: encodes to {len(ids)} tokens {ids}")
    return out
YES_MAP, NO_MAP = single_ids(YES_STRINGS), single_ids(NO_STRINGS)
YES_IDS, NO_IDS = sorted(set(YES_MAP.values())), sorted(set(NO_MAP.values()))
assert YES_IDS and NO_IDS and not set(YES_IDS) & set(NO_IDS)
print("\nFROZEN yes tokens:", YES_MAP, "\nFROZEN no tokens: ", NO_MAP)

## d. Prompt

The prompt is `f"{question} Answer yes or no."`, wrapped in the Qwen2-VL chat template with one image and `add_generation_prompt=True`. The model is loaded in bfloat16 (`DTYPE`); the legacy script used float16.

In [ ]:
PROMPT_TEMPLATE = "{question}" + PROMPT_SUFFIX
def make_prompt(question):
    return PROMPT_TEMPLATE.format(question=question)

def render_chat(question):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": make_prompt(question)}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(repr(render_chat(meta0.loc[0, "question"])))

## e. Model load

bf16 on CUDA, eval mode, gradients disabled.

In [ ]:
# Model load
from transformers import Qwen2VLForConditionalGeneration
torch_dtype = getattr(torch, DTYPE)
if DTYPE == "bfloat16" and torch.cuda.get_device_capability(0)[0] < 8:
    print("WARNING: this GPU has no native bfloat16 (e.g. T4). It may be slow or fail; consider DTYPE='float16'.")

model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch_dtype).to(DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
tok.padding_side = "left"          # index -1 is always the last prompt token

cfg_text = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
N_LAYERS_TOTAL = cfg_text.num_hidden_layers + 1       # embeddings + each transformer layer
HIDDEN_SIZE = cfg_text.hidden_size
print(f"Loaded {MODEL_ID} in {DTYPE}. hidden_states entries per token: {N_LAYERS_TOTAL} (embeddings + {cfg_text.num_hidden_layers} layers), hidden size {HIDDEN_SIZE}")
print("Existing cache for comparison: 29 x 1536 (read from the file in notebook 00).")

## f. Forced-choice scoring

One forward pass per question with `output_hidden_states=True`. Logits at the last prompt token give the yes/no probabilities, and `prediction = "yes"` iff `P(yes) > P(no)`.

Saved per question: `p_yes`, `p_no`, `p_yes_norm`, `margin` (log P(yes) - log P(no)), `binary_entropy` (bits), `vocab_entropy` (nats), `answer_mass`, `top1_token`, and `hidden_states[l][:, -1, :]` for every layer `l` (0 = embeddings).

In [ ]:
def score_logits(logits):
    logp = torch.log_softmax(logits.float(), dim=-1)
    lp_yes = torch.logsumexp(logp[:, YES_IDS], dim=-1)
    lp_no = torch.logsumexp(logp[:, NO_IDS], dim=-1)
    margin = lp_yes - lp_no
    p_yes_norm = torch.sigmoid(margin)
    q = p_yes_norm.clamp(1e-12, 1 - 1e-12)
    bin_ent = -(q * torch.log2(q) + (1 - q) * torch.log2(1 - q))
    vocab_ent = -(logp.exp() * logp).sum(-1)
    top1 = logp.argmax(-1)
    return dict(p_yes=lp_yes.exp().cpu(), p_no=lp_no.exp().cpu(), p_yes_norm=p_yes_norm.cpu(), margin=margin.cpu(),
                binary_entropy=bin_ent.cpu(), vocab_entropy=vocab_ent.cpu(), top1_id=top1.cpu())

def forward_batch(questions, images):
    # Returns (hidden [B, L, H] float32 on CPU, last-token logits [B, V])
    texts = [render_chat(q) for q in questions]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    hidden = torch.stack([h[:, -1, :] for h in out.hidden_states], dim=1).float().cpu()
    logits = out.logits[:, -1, :].float().cpu()
    return hidden, logits

def get_sample(row_idx):
    s = split_ds[int(row_idx)]
    return s["image"], s["question"]

## g/h. Extraction with checkpointing

Results are written in 100-question shards to `pope_full_shards/` (temp file, then rename). Finished rows are detected by dataset row index, so a re-run resumes. The final files are merged from the shards in the next cells.

The `split` column is assigned by image with a hash, so all questions of an image fall in the same split.

In [ ]:
def split_of(image_id):
    u = int(hashlib.md5(f"{SEED}-{image_id}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "train" if u < SPLIT_FRACS[0] else ("dev" if u < SPLIT_FRACS[0] + SPLIT_FRACS[1] else "test")

# selection and processing order
cand = meta0 if POPE_SETTINGS is None else meta0[meta0["category"].isin(POPE_SETTINGS)]
assert len(cand) > 0, f"No rows for POPE_SETTINGS={POPE_SETTINGS}"
order = cand["row_idx"].to_numpy().copy()
if SHUFFLE_ORDER:
    np.random.default_rng(SEED).shuffle(order)
if SUBSET_SIZE is not None:
    order = order[:SUBSET_SIZE]
SELECTED = [int(r) for r in order]
print(f"Selected {len(SELECTED)} questions (full run = {len(meta0)}).")

# run config (a resumed run must match)
RUN_CONFIG = dict(
    model_id=MODEL_ID, dtype=DTYPE, store_dtype=STORE_DTYPE,
    prompt_template=PROMPT_TEMPLATE, chat_rendering_example=render_chat("<QUESTION>"),
    yes_token_ids=YES_MAP, no_token_ids=NO_MAP,
    scoring="forced choice, logits at last prompt token, P(yes)=sum over yes ids, P(no)=sum over no ids; pred yes iff P(yes)>P(no)",
    token_position="last prompt token (index -1, left padding), all hidden_states entries",
    dataset=dict(name=DATASET_NAME, config=DATASET_CONFIG, local_path=DATASET_LOCAL_PATH, split=SPLIT_NAME,
                 fields=FIELDS, n_rows=len(meta0), settings_run=POPE_SETTINGS or sorted(meta0["category"].unique().tolist())),
    subset_size=SUBSET_SIZE, shuffle_order=SHUFFLE_ORDER, seed=SEED, batch_size=BATCH_SIZE,
    split_rule=f"md5('{{seed}}-{{image_source}}') -> train/dev/test fractions {SPLIT_FRACS}",
    n_hidden_state_entries=int(N_LAYERS_TOTAL), hidden_size=int(HIDDEN_SIZE),
    torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
)
import transformers
RUN_CONFIG["transformers"] = transformers.__version__
CONFIG_PATH = OUT / CONFIG_NAME
CHECK_KEYS = ["model_id", "dtype", "prompt_template", "yes_token_ids", "no_token_ids", "seed", "shuffle_order", "token_position"]
if CONFIG_PATH.exists():
    old = json.loads(CONFIG_PATH.read_text())
    bad = [k for k in CHECK_KEYS if old.get(k) != RUN_CONFIG.get(k)]
    assert not bad, f"Existing run in {OUT} used different settings for {bad}. Use a new OUT_DIR or delete the old shards."
    print("Resuming a previous run (config matches).")
RUN_CONFIG.update(status="running", started_utc=datetime.datetime.utcnow().isoformat() + "Z")
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2))

# shard helpers
def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def list_shards():
    return sorted(SHARD_DIR.glob("shard_*.pt"))

def load_done():
    done, next_k = set(), 0
    for p in list_shards():
        next_k = max(next_k, int(p.stem.split("_")[1]) + 1)
        try:
            blob = torch.load(p, map_location="cpu", weights_only=False)
            done.update(int(r) for r in blob["row_idx"])
        except Exception as e:
            print(f"  corrupt shard {p.name} ({e}); deleting, its rows will be redone")
            p.unlink()
    return done, next_k

def make_records(row_ids, questions, scores):
    recs = []
    for i, r in enumerate(row_ids):
        m = meta0.loc[r]
        p_yes, p_no = float(scores["p_yes"][i]), float(scores["p_no"][i])
        pred = "yes" if p_yes > p_no else "no"
        recs.append(dict(
            row_idx=int(r), id=m["id"], question_id=m["question_id"], image_id=m["image_source"], setting=m["category"],
            question=questions[i], gt_answer=m["answer"], pred_answer=pred,
            hallucination=int(pred == "yes" and m["answer"] == "no"),
            p_yes=p_yes, p_no=p_no, p_yes_norm=float(scores["p_yes_norm"][i]), margin=float(scores["margin"][i]),
            binary_entropy=float(scores["binary_entropy"][i]), vocab_entropy=float(scores["vocab_entropy"][i]),
            answer_mass=p_yes + p_no, top1_token=tok.decode([int(scores["top1_id"][i])]),
            split=split_of(m["image_source"]), prompt=make_prompt(questions[i]),
        ))
    return recs

def run_chunk(chunk):
    feats, recs = [], []
    for b in range(0, len(chunk), BATCH_SIZE):
        sub = chunk[b:b + BATCH_SIZE]
        got = [get_sample(r) for r in sub]
        images, questions = [g[0] for g in got], [g[1] for g in got]
        hidden, logits = forward_batch(questions, images)
        feats.append(hidden)
        recs += make_records(sub, questions, score_logits(logits))
    feats = torch.cat(feats)
    assert torch.isfinite(feats).all(), "non-finite hidden states"
    assert feats.abs().max() < 65000 or STORE_DTYPE != "float16", "hidden state magnitude would overflow float16; set STORE_DTYPE='float32' or 'bfloat16'"
    return dict(row_idx=torch.tensor(chunk), features=feats.to(getattr(torch, STORE_DTYPE)), meta=recs)

In [ ]:
# Main loop (safe to interrupt and re-run)
from tqdm.auto import tqdm
done, next_k = load_done()
todo = [r for r in SELECTED if r not in done]
print(f"Already done: {len(done & set(SELECTED))} | to do: {len(todo)}")
t0 = time.time()
pbar = tqdm(total=len(todo))
for s in range(0, len(todo), CHECKPOINT_EVERY):
    chunk = todo[s:s + CHECKPOINT_EVERY]
    atomic_save(run_chunk(chunk), SHARD_DIR / f"shard_{next_k:05d}.pt")
    next_k += 1
    pbar.update(len(chunk))
pbar.close()
print(f"Extraction finished in {(time.time() - t0) / 60:.1f} min")

## Merge shards into the final files

In [ ]:
def merge_shards():
    blobs = [torch.load(p, map_location="cpu", weights_only=False) for p in list_shards()]
    sel = set(SELECTED)
    rows, feats, recs, seen = [], [], [], set()
    for b in blobs:
        for j, r in enumerate(b["row_idx"].tolist()):
            if r in seen or r not in sel:
                continue
            seen.add(r)
            rows.append(r); feats.append(b["features"][j]); recs.append(b["meta"][j])
    perm = np.argsort(rows)
    row_idx = torch.tensor([rows[i] for i in perm])
    features = torch.stack([feats[i] for i in perm])
    meta = pd.DataFrame([recs[i] for i in perm])
    return row_idx, features, meta

row_idx, features, meta = merge_shards()
print("merged:", tuple(features.shape), features.dtype, "| metadata rows:", len(meta))
atomic_save(dict(features=features, row_idx=row_idx, token_position="last_prompt_token",
                 note="features[i] corresponds to metadata row with row_idx[i]; shape [N, n_hidden_state_entries, hidden]"),
            OUT / FEATURES_NAME)
try:
    meta.to_parquet(OUT / META_NAME, index=False)
    META_PATH = OUT / META_NAME
except Exception as e:
    META_PATH = OUT / META_NAME.replace(".parquet", ".csv")
    print("parquet unavailable (", e, ") -> writing CSV")
    meta.to_csv(META_PATH, index=False)
RUN_CONFIG.update(status="complete" if len(meta) == len(SELECTED) else "partial", n_rows_saved=int(len(meta)),
                  finished_utc=datetime.datetime.utcnow().isoformat() + "Z", date=str(datetime.date.today()))
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2))
print("wrote", OUT / FEATURES_NAME, META_PATH, CONFIG_PATH)

## i. Verification

A full run of 9,000 questions should give a features file of about 0.8 GB (9,000 x 29 x 1536 x 2 bytes). The cell flags anything far off.

In [ ]:
blob = torch.load(OUT / FEATURES_NAME, map_location="cpu", weights_only=True)
F, R = blob["features"], blob["row_idx"].numpy()
M = pd.read_parquet(META_PATH) if str(META_PATH).endswith(".parquet") else pd.read_csv(META_PATH)
print("features:", tuple(F.shape), F.dtype, "| metadata:", M.shape)
assert len(M) == F.shape[0] == len(R) and (M["row_idx"].to_numpy() == R).all(), "features/metadata misaligned"
missing = sorted(set(SELECTED) - set(R.tolist()))
print("rows selected:", len(SELECTED), "| rows saved:", len(R), "| missing:", len(missing), missing[:10])
print("duplicate row_idx:", int(pd.Series(R).duplicated().sum()), "| non-finite values:", int((~torch.isfinite(F.float())).sum()))
print("max |hidden state| per kept entry (head):", F.float().abs().amax(dim=(0, 2))[:5].tolist(), "... overall max", float(F.float().abs().max()))

size_gb = (OUT / FEATURES_NAME).stat().st_size / 1e9
expected_gb = len(R) * F.shape[1] * F.shape[2] * F.element_size() / 1e9
print(f"\nfeature file size: {size_gb:.3f} GB | expected for {len(R)} rows: {expected_gb:.3f} GB | for 9,000 rows ~0.80 GB")
if not (0.9 * expected_gb <= size_gb <= 1.2 * expected_gb):
    print("!!! FLAG: file size is far from expectation, investigate before using the cache")

print("\nprediction distribution:", M["pred_answer"].value_counts().to_dict())
print("answer_mass: mean %.3f | share < 0.9: %.3f" % (M["answer_mass"].mean(), (M["answer_mass"] < 0.9).mean()))
print("top-1 token counts (head):", M["top1_token"].value_counts().head(6).to_dict())
M["correct"] = (M["pred_answer"] == M["gt_answer"]).astype(int)
g = M.groupby("setting")
summary = pd.DataFrame({
    "n": g.size(),
    "accuracy": g["correct"].mean(),
    "yes_rate": g["pred_answer"].apply(lambda s: (s == "yes").mean()),
    "false_yes_rate (pred yes | gt no)": M[M.gt_answer == "no"].groupby("setting")["hallucination"].mean(),
    "hallucination mean over all rows": g["hallucination"].mean(),
}).round(4)
print("\n", summary)
print("\naccuracy overall: %.4f | false-yes rate overall: %.4f" % (M["correct"].mean(), M[M.gt_answer == "no"]["hallucination"].mean()))
print("split sizes (rows):", M["split"].value_counts().to_dict(), "| images per split:", M.groupby("split")["image_id"].nunique().to_dict())
assert M.groupby("image_id")["split"].nunique().max() == 1, "an image appears in two splits"

## Download (optional)
Preferred route is Google Drive (`USE_DRIVE = True`). Otherwise set `DOWNLOAD_AT_END = True` and re-run this cell.

In [ ]:
if DOWNLOAD_AT_END:
    from google.colab import files
    for f in [CONFIG_PATH, META_PATH, OUT / FEATURES_NAME]:
        files.download(str(f))
if not KEEP_SHARDS:
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Done. Copy", FEATURES_NAME, META_PATH.name, CONFIG_NAME, "into the repo's results/cached_features/ (do NOT commit the .pt).")